In [ ]:
import string
import os
import random
import re
import webbrowser
import urllib.parse
import speech_recognition as sr
import wikipedia
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
wikipedia.set_lang("en")

def speak(text):
    print(f"JARVIS: {text}")
    safe_text = text.replace("'", "")
    cmd = f'powershell -Command "Add-Type -AssemblyName System.Speech; $speak = New-Object System.Speech.Synthesis.SpeechSynthesizer; $speak.Speak(\'{safe_text}\')"'
    os.system(cmd)

useful_facts = [
    "The human brain consumes about 20 percent of the total energy of the body, although it makes up only 2 percent of its mass.",
    "The Word2Vec vector model turns words into numbers so that concepts with similar meanings are located close to each other in a multidimensional space.",
    "The oldest university known today is the University of Bologna, founded in 1088.",
    "Ada Lovelace is considered the first programmer in history: in 1843 she wrote an algorithm for the Analytical Engine of Charles Babbage.",
    "The speed of light in a vacuum is approximately 300 thousand kilometers per second.",
    "Graphene is the strongest material known to science."
]

knowledge_base = {
    "hello hi hey good morning good evening": "Hello! O great master, I am glad to serve you.",
    "how are you how is it going how do you feel": "Praise be to the gods and to your prayers, I am doing great! Ready to work.",
    "who are you what can you do": "I am your humble servant, most radiant master! I can search for information on Wikipedia, open YouTube and share facts.",
    "what is artificial intelligence ai neural network": "Artificial intelligence is a field of computer science that creates systems capable of solving tasks that require human intelligence.",
    "how does the internet work network protocols": "The internet works by transmitting data packets over TCP/IP protocols between routers and servers.",
    "what is a database sql": "A database is an organized system for storing information, and SQL is a language for running queries against it.",
    "what is the weather like outside": "Thanks to your smile, O beautiful master, the whole world has dimmed!",
    "thank you thanks": "Do not thank me, O great mind, I am a humble servant and will help with whatever I can!",
    "bye goodbye see you later turn off": "Bye, farewell my beautiful radiant master!"
}

# Question words are important for matching questions, so they are not removed
keep_words = {"how", "what", "who", "you", "are", "can"}
my_stopwords = set(stopwords.words('english')) - keep_words

def preprocess_text(text):
    tokens = word_tokenize(text.lower(), language='english')
    cleaned = [word for word in tokens if word not in string.punctuation and word not in my_stopwords]
    return " ".join(cleaned)

def get_best_response(user_input):
    cleaned_input = preprocess_text(user_input)
    if not cleaned_input:
        return "Sorry, I did not quite understand you."

    questions = list(knowledge_base.keys())
    cleaned_questions = [preprocess_text(q) for q in questions]

    vectorizer = TfidfVectorizer().fit(cleaned_questions + [cleaned_input])
    q_vectors = vectorizer.transform(cleaned_questions)
    input_vector = vectorizer.transform([cleaned_input])

    similarities = cosine_similarity(input_vector, q_vectors)[0]
    best_match_index = similarities.argmax()

    if similarities[best_match_index] < 0.2:
        return "I do not know the answer to that question yet, but you can ask me to look it up on Wikipedia!"

    matched_question = questions[best_match_index]
    return knowledge_base[matched_question]

def clean_wikipedia_query(text):
    stop_phrases = [
        r'\bfind\b', r'\bsearch\b', r'\blook up\b', r'\bopen\b',
        r'\bon wikipedia\b', r'\bin wikipedia\b', r'\bwikipedia\b',
        r'\babout\b', r'\btell me\b', r'\bfor\b', r'\binformation\b', r'\barticle\b'
    ]
    query = text.lower()
    for pattern in stop_phrases:
        query = re.sub(pattern, '', query)
    return " ".join(query.split())

def open_wikipedia_page(query):
    try:
        search_results = wikipedia.search(query)
        if search_results:
            best_match = search_results[0]
            page = wikipedia.page(best_match, auto_suggest=False)
            speak(f"Found the article: {page.title}")
            webbrowser.open(page.url)
        else:
            speak("Unfortunately, I found nothing for this query.")
    except Exception:
        encoded_query = urllib.parse.quote(query)
        speak(f"Searching Wikipedia for: {query}")
        webbrowser.open(f"https://en.wikipedia.org/w/index.php?search={encoded_query}")

recognizer = sr.Recognizer()

def listen():
    with sr.Microphone() as source:
        print("\nListening...")
        recognizer.dynamic_energy_threshold = False
        recognizer.energy_threshold = 300

        recognizer.pause_threshold = 1.8        
        recognizer.non_speaking_duration = 0.8
        
        recognizer.adjust_for_ambient_noise(source, duration=0.8)
        try:
            audio = recognizer.listen(source, timeout=6, phrase_time_limit=10)
            text = recognizer.recognize_google(audio, language='en-US')
            print(f"You: {text}")
            return text
        except (sr.WaitTimeoutError, sr.UnknownValueError):
            return None
        except sr.RequestError:
            speak("Speech recognition service error.")
            return None

speak("System activated. How can I help?")

while True:
    user_command = listen()
    
    if user_command:
        cmd_lower = user_command.lower()
        
        if "wikipedia" in cmd_lower:
            search_query = clean_wikipedia_query(cmd_lower)
            if search_query:
                open_wikipedia_page(search_query)
            else:
                speak("Opening the Wikipedia main page.")
                webbrowser.open("https://en.wikipedia.org")
        elif "youtube" in cmd_lower:
            speak("Opening YouTube.")
            webbrowser.open("https://www.youtube.com")
        elif "fact" in cmd_lower or "tell me something" in cmd_lower:
            speak(random.choice(useful_facts))
        else:
            response = get_best_response(user_command)
            speak(response)
        
        if any(word in cmd_lower for word in ["turn off", "bye", "goodbye"]):
            break
